# 03 — Input sources

All three built-in sources yield the same `Segment(file_id, channel, spkid, start, end, lang?, audio_ref)` type and plug into the same `extract()` driver:

| Source | Picks up | Audio |
|---|---|---|
| `plain_text` | TSV manifest + `wav.scp` | `wav.scp` paths or pipes |
| `kaldi` | A Kaldi data dir (`wav.scp`, `segments`, `utt2spk`, …) | `wav.scp` paths or pipes |
| `hf` | A HuggingFace dataset (on disk or hub) | dataset `audio` column OR `wav.scp` fallback |

This notebook builds a tiny instance of each and runs `extract()` with the offline fake encoder.

In [ ]:
import tempfile, wave, hashlib
from pathlib import Path
import numpy as np

from sonar_probe import ExtractConfig, extract, load_store
from sonar_probe.sources import get_source
from sonar_probe.sinks import get_sink
from sonar_probe.engine.speech import SpeechEncoderPool

WORK = Path(tempfile.mkdtemp(prefix="sonar-probe-sources-"))
print("workdir:", WORK)

DIM = 1024
def _hv(k): 
    rng = np.random.default_rng(int.from_bytes(hashlib.sha256(k).digest()[:8], "big"))
    v = rng.standard_normal(DIM).astype(np.float32); return v / (np.linalg.norm(v) + 1e-9)

class FakeSpeechEncoder:
    def encode(self, waves): return np.stack([_hv(w.tobytes()) for w in waves]).astype(np.float32)

fake_pool = SpeechEncoderPool(device="cpu", encoder_factory=lambda l, d: FakeSpeechEncoder())

In [ ]:
SR = 16000
def tone(path, hz, seconds):
    t = np.arange(int(SR * seconds)) / SR
    x = (0.3 * np.sin(2 * np.pi * hz * t) * 32767).astype(np.int16)
    with wave.open(str(path), "wb") as w:
        w.setnchannels(1); w.setsampwidth(2); w.setframerate(SR); w.writeframes(x.tobytes())

audio = WORK / "audio"; audio.mkdir()
tone(audio / "rec01.wav", 440.0, 6.0)
tone(audio / "rec02.wav", 880.0, 6.0)

## 1. `plain_text` — TSV manifest

Columns: `file_id`, `channel`, `spkid`, `start`, `end` (optional `lang`). Channel labels `A`/`B`/`L`/`R` are coerced; numeric channels work too. Pipe entries in `wav.scp` (trailing `|`) are decoded via subprocess.

In [ ]:
pt_dir = WORK / "plain_text"; pt_dir.mkdir()
(pt_dir / "wav.scp").write_text(f"rec01 {audio/'rec01.wav'}\nrec02 {audio/'rec02.wav'}\n")
(pt_dir / "segments.tsv").write_text(
    "file_id\tchannel\tspkid\tstart\tend\tlang\n"
    "rec01\t0\tspk1\t0.0\t2.0\teng\n"
    "rec01\t0\tspk1\t2.0\t4.0\teng\n"
    "rec02\t0\tspk2\t0.0\t3.0\teng\n"
)

src = get_source("plain_text", input=pt_dir / "segments.tsv", wav_scp=pt_dir / "wav.scp")
snk = get_sink("kaldi_ark", output=WORK / "store_plain_text")
res = extract(src, snk, ExtractConfig(device="cpu", batch_size=4, speech_lang="eng"), pool=fake_pool)
print(f"plain_text: written={res.written}, ids={load_store(WORK/'store_plain_text').ids()}")

## 2. `kaldi` — Kaldi data directory

A standard Kaldi data dir: `wav.scp`, `segments`, `utt2spk`, optional `reco2file_and_channel`, optional `utt2lang`. Each `utt_id` becomes the segment's `segment_id`.

In [ ]:
kd = WORK / "kaldi_data"; kd.mkdir()
(kd / "wav.scp").write_text(f"rec01 {audio/'rec01.wav'}\nrec02 {audio/'rec02.wav'}\n")
(kd / "segments").write_text(
    "utt01 rec01 0.0 2.0\n"
    "utt02 rec01 2.0 4.0\n"
    "utt03 rec02 0.0 3.0\n"
)
(kd / "utt2spk").write_text("utt01 spk1\nutt02 spk1\nutt03 spk2\n")
(kd / "utt2lang").write_text("utt01 eng\nutt02 eng\nutt03 eng\n")

src = get_source("kaldi", input=kd)
snk = get_sink("kaldi_ark", output=WORK / "store_kaldi")
res = extract(src, snk, ExtractConfig(device="cpu", batch_size=4), pool=fake_pool)
print(f"kaldi: written={res.written}, ids={load_store(WORK/'store_kaldi').ids()}")

## 3. `hf` — HuggingFace dataset with audio column

Loads via `datasets.load_from_disk` (if `input` is a directory) or `datasets.load_dataset` otherwise. Required columns: `file_id`, `channel`, `spkid`, `start`, `end` (optional `lang`). If the dataset has an `audio` column it is used directly; otherwise pass `wav_scp=` to resolve `file_id` to a path.

In [ ]:
from datasets import Dataset, Audio

ds = Dataset.from_dict({
    "file_id": ["rec01", "rec01", "rec02"],
    "channel": [0, 0, 0],
    "spkid":  ["spk1", "spk1", "spk2"],
    "start":  [0.0, 2.0, 0.0],
    "end":    [2.0, 4.0, 3.0],
    "lang":   ["eng", "eng", "eng"],
    "audio":  [str(audio/"rec01.wav"), str(audio/"rec01.wav"), str(audio/"rec02.wav")],
}).cast_column("audio", Audio(sampling_rate=16000))
ds.save_to_disk(str(WORK / "hf_dataset"))

src = get_source("hf", input=WORK / "hf_dataset")
snk = get_sink("hf_dataset", output=WORK / "store_hf")
res = extract(src, snk, ExtractConfig(device="cpu", batch_size=4), pool=fake_pool)
print(f"hf: written={res.written}, ids={load_store(WORK/'store_hf').ids()}")

## HF without an audio column

Drop the `audio` column and pass `wav_scp=`; the source falls back to resolving `file_id → path`.

In [ ]:
ds2 = ds.remove_columns(["audio"])
ds2.save_to_disk(str(WORK / "hf_no_audio"))
wscp = WORK / "hf_no_audio_wav.scp"
wscp.write_text(f"rec01 {audio/'rec01.wav'}\nrec02 {audio/'rec02.wav'}\n")

src = get_source("hf", input=WORK / "hf_no_audio", wav_scp=wscp)
snk = get_sink("hf_dataset", output=WORK / "store_hf2")
res = extract(src, snk, ExtractConfig(device="cpu", batch_size=4), pool=fake_pool)
print(f"hf (no audio col): written={res.written}, ids={load_store(WORK/'store_hf2').ids()}")

## Compare across sources

All four stores contain embeddings for the same waveforms, so same-segment IDs match (modulo IDs chosen by the source):

In [ ]:
for name in ["store_plain_text", "store_kaldi", "store_hf", "store_hf2"]:
    s = load_store(WORK / name)
    print(f"{name:22s} count={len(s)} first-id={next(iter(s.ids()))}")